# 🎀 공주비서 — vLLM으로 Qwen3.5 서빙하기 (1단계)

> 12주차 과제 ① *"개인 프로젝트에 쓰이는 공개 가중치 모델을 vLLM으로 서빙해 보기"*
> 실행 환경: **Colab Pro** · 브랜치 `week12`

## 이번 노트북의 목표

**vLLM 서버를 띄우고 첫 응답을 받아내는 것까지.** 딱 거기까지다.
정확도 측정·처리량 비교는 2·3단계 노트북에서 한다.

## 왜 이걸 하나 — 봇과의 관계

지금 공주비서의 뇌는 **Claude API**다. 가중치가 비공개라 내가 직접 서빙할 수 없다.
그래서 "모델을 서빙한다"를 경험하려면 **공개 가중치 모델**을 따로 데려와야 한다.

```
지금:   Discord → [Claude API] → 노션
목표:   Discord → [vLLM Qwen3.5 = 1차 라우터] → (복잡하면) Claude → 노션
                  ↑ 이 노트북에서 만드는 것
```

라우터란: "사진 올렸어요" 같은 쉬운 분류는 **로컬 모델이 0원에** 처리하고,
애매한 판단만 Claude에 넘기는 구조. 분류는 작은 모델로 충분하다.

⚠️ **이번 주엔 봇 코드를 건드리지 않는다.** Claude API가 잘 돌고 있으므로
연결은 나중에 별도 브랜치에서. 이 노트북은 봇과 독립적으로 돈다.

## vLLM이 뭘 해주나

`transformers`로도 모델을 돌릴 수는 있다. vLLM이 추가로 주는 것:

| | transformers | vLLM |
|---|---|---|
| 인터페이스 | 파이썬 함수 호출 | **OpenAI 호환 HTTP 서버** |
| 동시 요청 | 하나씩 | **연속 배칭**(continuous batching) |
| KV 캐시 | 요청마다 통째로 잡음 | **PagedAttention** — 페이지 단위로 쪼개 재사용 |

특히 첫 줄이 중요하다. OpenAI 호환 서버라서, 나중에 봇에 붙일 때
**`base_url`만 바꾸면** 된다. 봇 구조를 뒤집지 않아도 된다.

---
## 0. GPU 확인

vLLM은 **GPU가 필수**다. (지난 학기 EC2 t3.micro로는 불가능했던 이유가 이것.)

Colab 메뉴 → **런타임 → 런타임 유형 변경 → GPU** 를 먼저 켜고 아래를 실행한다.

In [3]:
!nvidia-smi

Tue Jul 28 03:51:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   45C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 뭘 봐야 하나

출력의 **GPU 이름**과 **메모리 총량**을 확인한다. Colab Pro에서 주로 나오는 것:

| GPU | 메모리 | Qwen3.5-4B 적합도 |
|---|---|---|
| **L4** | 24GB | 넉넉함 ✅ (이 노트북 기준) |
| **A100** | 40GB | 아주 넉넉 ✅ |
| **T4** | 16GB | 가능하지만 `--dtype half` 필요 ⚠️ |

**T4가 걸렸다면** 주의: T4는 `bfloat16`을 하드웨어로 지원하지 않는다.
아래 서버 실행 셀에서 `--dtype half`(=float16)를 반드시 추가해야 한다.
L4/A100이면 그대로 두면 된다.

---
## 1. vLLM 설치

Qwen3.5는 2026년 2월에 나온 신형이라, **모델 카드가 vLLM nightly(main 브랜치)를 요구한다.**
정식 릴리스 버전엔 이 아키텍처(Gated DeltaNet + sparse MoE)가 아직 안 들어있을 수 있다.

⚠️ **설치 후 런타임 재시작이 거의 확실하다.** vLLM이 자기 버전의 torch를 끌고 오는데,
Colab에 이미 로드된 torch와 충돌하기 때문이다. 재시작하라는 안내가 뜨면 재시작하고,
**0번 셀부터가 아니라 2번(서버 실행)부터** 이어서 실행하면 된다. (설치는 디스크에 남는다)

In [4]:
# uv = 훨씬 빠른 pip. Colab엔 기본으로 없어서 먼저 깐다.
!pip install -q uv

# nightly 휠 저장소를 추가로 바라보게 해서 vLLM 최신판을 설치한다.
# --torch-backend=auto : 이 GPU에 맞는 CUDA 빌드를 알아서 고른다.
!uv pip install --system vllm --torch-backend=auto \
    --extra-index-url https://wheels.vllm.ai/nightly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 86.2 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.13 environment at: /usr
Resolved 193 packages in 24.14s                                      
Prepared 99 packages in 46.18s                                           
Uninstalled 14 packages in 724ms
Installed 99 packages in 268ms                              
 + anthropic==0.120.0
 + apache-tvm-ffi==0.1.10
 + astor==0.8.1
 + blake3==1.0.9
 + cbor2==6.1.3
 + compressed-tensors==0.17.0
 - cuda-bindings==12.9.7
 + cuda-bindings==13.3.1
 - cuda-core==0.3.2
 + cuda-core==1.0.1
 - cuda-python==12.9.7
 + cuda-python==13.3.1
 + cuda-tile==1.5.0
 - cuda-toolkit==12.8.1
 + cuda-toolkit==13.2.1
 + depyf==0.20.0
 + detect-installer==0.1.0
 + dnspython==2.8.0
 + email-validator==2.3.0
 - fastapi==0.139.0
 + fastapi==0.136.3
 + fastapi-cli==0.0.32
 + fastapi-cloud-cli==0.22.2
 + fastar==0.11.0
 + fastsafetensors==0.3.3
 + flashinfer-python==0.6.15.post1
 + httpx-sse==0.4.3
 + humming-kernels

In [5]:
# 설치 확인. 버전 문자열에 dev/nightly 냄새가 나면 정상.
import vllm
print("vLLM", vllm.__version__)

vLLM 0.23.1rc1.dev1458+ge222c33f2


### ⚠️ 필수 응급처치 — torchaudio 제거

**실제로 여기서 막혔다 (2026-07-28).** 증상:

```
RuntimeError: Detected that PyTorch and TorchAudio were compiled with
different CUDA versions. PyTorch has CUDA version 13.2 whereas
TorchAudio has CUDA version 12.8.
```

무슨 일이냐면:

```
vLLM nightly가 끌고 온 torch   → CUDA 13.2
Colab에 원래 깔려 있던 torchaudio → CUDA 12.8   ← 안 맞음
```

vLLM은 `transformers`를 import하고, transformers는 `loss_rnnt.py`에서 `torchaudio`를
import한다. 거기서 위 에러가 터져 **vLLM이 시작도 못 하고 죽는다.**

핵심은 이것이다 — **transformers는 torchaudio가 *없는* 경우는 대비하지만
*깨진* 경우는 대비하지 않는다.** 없으면 `ImportError`가 나고 조용히 넘어가게
돼 있는데, 지금은 `RuntimeError`라 그 방어망을 뚫고 나온다.

→ 그래서 **지워버리면 해결된다.** 우리는 텍스트 모델만 쓰니 torchaudio는 필요 없다.

In [6]:
# torchaudio는 텍스트 서빙에 불필요하다. CUDA 버전 충돌만 일으키므로 제거한다.
!pip uninstall -y torchaudio

# 제거 후 vLLM이 정상 import 되는지 확인 (여기서 에러가 없어야 서버가 뜬다)
!python -c "from vllm.engine.arg_utils import EngineArgs; print('✅ vLLM import OK')"

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
✅ vLLM import OK


> 💡 `vllm serve`는 **별도 프로세스**로 뜨기 때문에, 이 제거 후 런타임을 재시작할
> 필요 없이 아래 서버 셀로 바로 넘어가면 된다.

### 🔧 그래도 안 되면 (폴백)

nightly는 말 그대로 매일 빌드되는 물건이라 가끔 깨져 있다. 순서대로 시도한다.

**폴백 1 — torch를 Colab에 맞춰 고정.** CUDA 13.2를 끌고 오는 게 문제였으니
`--torch-backend=auto` 대신 Colab과 같은 계열로 못박는다:

```python
!uv pip install --system vllm --torch-backend=cu128 \
    --extra-index-url https://wheels.vllm.ai/nightly
```

**폴백 2 — 정식 버전 + 한 세대 전 모델.** 여기까지 오면 nightly를 포기한다.
과제 목적(vLLM 서빙 경험)은 그대로 달성된다.

```python
!uv pip install --system -U vllm
```
그리고 아래 `MODEL`을 `"Qwen/Qwen3-4B"`로 바꾼다.
(`--reasoning-parser qwen3`는 그대로 두면 된다.)

---
## 2. 서버 띄우기 — 여기가 이 노트북의 핵심

### 왜 `!vllm serve`를 그냥 쓰면 안 되나

`vllm serve`는 **서버**다. 실행하면 종료되지 않고 계속 떠 있는다.
노트북 셀에서 `!vllm serve ...`를 치면 **그 셀이 영원히 안 끝나고**,
다음 셀을 실행할 수 없다. 서버는 떴는데 요청을 보낼 방법이 없는 상태가 된다.

그래서 **백그라운드 프로세스로 띄우고 로그는 파일로 뺀다.**
`subprocess.Popen`이 그 역할이다. 셀은 바로 끝나고 서버는 뒤에서 계속 돈다.

### 옵션 설명 — 모델 카드를 그대로 베끼면 안 되는 이유

모델 카드의 권장 명령은 이렇다:

```
vllm serve Qwen/Qwen3.5-4B --port 8000 --tensor-parallel-size 1 \
    --max-model-len 262144 --reasoning-parser qwen3
```

여기서 **`--max-model-len 262144`(26만 토큰)를 그대로 쓰면 거의 확실히 OOM**이 난다.
vLLM은 시작할 때 그 길이만큼의 **KV 캐시를 GPU에 미리 잡아두기** 때문이다.
26만 토큰짜리 캐시는 모델 가중치보다 훨씬 큰 메모리를 요구한다.

우리 용도(라우터 = 짧은 문장 분류)엔 **8192면 차고 넘친다.**
이게 vLLM을 직접 만져봐야 알게 되는 종류의 지식이다.

| 옵션 | 값 | 이유 |
|---|---|---|
| `--max-model-len` | `8192` | 카드 권장 262144은 KV 캐시 OOM. 라우터엔 과분 |
| `--gpu-memory-utilization` | `0.90` | GPU의 90%까지 사용 허용 |
| `--reasoning-parser qwen3` | | thinking 출력을 별도 필드로 분리해준다 |
| `--tensor-parallel-size` | `1` | GPU 1장이니까 1 |

In [7]:
import subprocess, os, sys

MODEL = "Qwen/Qwen3.5-4B"   # 폴백 시 "Qwen/Qwen3-4B"
PORT  = 8000
LOG   = "/content/vllm.log"

cmd = [
    "vllm", "serve", MODEL,
    "--port", str(PORT),
    "--tensor-parallel-size", "1",
    "--max-model-len", "8192",          # 카드 권장 262144 → OOM. 위 설명 참고
    "--gpu-memory-utilization", "0.90",
    "--reasoning-parser", "qwen3",
    # ⚠️ T4를 받았다면 아래 줄의 주석을 풀 것 (T4는 bfloat16 미지원)
    # "--dtype", "half",
]

# 로그를 파일로 빼고 백그라운드로 띄운다 → 셀은 즉시 끝나고 서버는 계속 돈다.
logf = open(LOG, "w")
server = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
print("서버 프로세스 시작됨. PID =", server.pid)
print("모델 다운로드 + 로딩에 몇 분 걸립니다. 다음 셀에서 기다립니다.")

서버 프로세스 시작됨. PID = 2503
모델 다운로드 + 로딩에 몇 분 걸립니다. 다음 셀에서 기다립니다.


### 준비될 때까지 기다리기 — 지난주에 배운 그것

여기서 **11주차 CI/CD에서 만났던 함정이 똑같이 재현된다.**

그때: 배포 직후 `curl /health`를 한 번만 쐈다가 `Connection reset by peer`로 실패.
앱이 아직 안 뜬 상태였다. → **뜰 때까지 재시도**하는 루프로 고쳤다. (readiness)

지금도 완전히 같다. `Popen`은 **즉시** 리턴하지만 vLLM은 그 시점에 아직
모델을 다운로드조차 안 끝냈다. 수 GB를 받고 GPU에 올리는 데 몇 분이 걸린다.
바로 요청을 보내면 연결 거부다.

그래서 같은 해법을 쓴다 — **살아날 때까지 폴링**한다.
서버가 준비되면 `/v1/models`가 200을 준다.

In [8]:
import time, urllib.request, urllib.error

BASE = f"http://localhost:{PORT}"
DEADLINE = time.time() + 900          # 최대 15분 대기
ready = False

while time.time() < DEADLINE:
    # 서버 프로세스가 죽었으면 기다려봐야 소용없다 — 즉시 로그를 보여주고 멈춘다.
    if server.poll() is not None:
        print("\n❌ 서버 프로세스가 죽었습니다. 로그 마지막 부분:\n")
        print(open(LOG).read()[-4000:])
        break

    try:
        with urllib.request.urlopen(f"{BASE}/v1/models", timeout=3) as r:
            if r.status == 200:
                ready = True
                print("\n✅ 서버 준비 완료")
                break
    except Exception:
        pass                          # 아직 안 떴다 — 정상. 계속 기다린다.

    print(".", end="")
    time.sleep(5)

if not ready:
    print("\n서버가 준비되지 않았습니다. 위 로그를 확인하세요.")
    print("CUDA 버전 충돌(torchaudio) 에러라면 1절의 '응급처치' 셀을 실행하고 "
          "이 서버 셀부터 다시 실행하세요.")

................................................................................................
✅ 서버 준비 완료


In [9]:
# 기동 로그 훑어보기. vLLM이 KV 캐시를 얼마나 잡았는지가 여기 찍힌다.
# "GPU KV cache size" / "Maximum concurrency" 줄을 눈여겨볼 것.
log = open(LOG).read()
for line in log.splitlines():
    if any(k in line for k in ("KV cache", "concurrency", "Loading", "memory")):
        print(line)

(APIServer pid=2503) INFO 07-28 03:53:34 [api_utils.py:273] non-default args: {'model_tag': 'Qwen/Qwen3.5-4B', 'model': 'Qwen/Qwen3.5-4B', 'max_model_len': 8192, 'reasoning_parser': 'qwen3', 'gpu_memory_utilization': 0.9}
Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:01<00:01,  1.35s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:02<00:00,  1.29s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:02<00:00,  1.30s/it]
(EngineCore pid=3049) INFO 07-28 03:55:35 [default_loader.py:430] Loading weights took 2.68 seconds
(EngineCore pid=3049) INFO 07-28 03:55:36 [gpu_model_runner.py:5397] Model loading took 8.61 GiB memory and 30.524691 seconds
(EngineCore pid=3049) INFO 07-28 04:00:10 [gpu_model_runner.py:6659] Profiling CUDA graph memory: PIECEWISE=51 (largest=512), FULL=35 (largest=256)
(EngineCore pid=3049) INFO 07-28 04:00:14 [gpu_model_runner.py:6784] Est

---
## 3. 첫 응답 받기

서버가 떴으니 이제 **OpenAI와 똑같은 형식**으로 말을 걸 수 있다.
먼저 어떤 모델이 물려 있는지 확인한다.

In [10]:
!curl -s http://localhost:8000/v1/models | python3 -m json.tool

{
    "object": "list",
    "data": [
        {
            "id": "Qwen/Qwen3.5-4B",
            "object": "model",
            "created": 1785211275,
            "owned_by": "vllm",
            "root": "Qwen/Qwen3.5-4B",
            "parent": null,
            "max_model_len": 8192,
            "permission": [
                {
                    "id": "modelperm-bb2b6469e629dcae",
                    "object": "model_permission",
                    "created": 1785211275,
                    "allow_create_engine": false,
                    "allow_sampling": true,
                    "allow_logprobs": true,
                    "allow_search_indices": false,
                    "allow_view": true,
                    "allow_fine_tuning": false,
                    "organization": "*",
                    "group": null,
                    "is_blocking": false
                }
            ]
        }
    ]
}


### 채팅 요청 — 그리고 thinking 모드 함정

Qwen3.5는 **thinking(추론) 모드가 기본으로 켜져 있다.** 답하기 전에 스스로
길게 생각하는 과정을 먼저 뱉는다. 어려운 문제엔 좋지만 **라우터엔 최악**이다:
"이건 사진 인증이다" 한 마디 하는 데 수백 토큰을 쓰고 몇 초를 잡아먹는다.

끄는 법: 요청 본문에 `"chat_template_kwargs": {"enable_thinking": false}`.
아래에서 **켠 것과 끈 것을 나란히** 재본다. 이 차이가 2단계 라우터 설계의 근거가 된다.

In [11]:
import json, time, urllib.request

def chat(prompt, thinking=True, max_tokens=512):
    # vLLM 서버에 한 번 물어보고 (답, 걸린시간, 생성토큰수)를 돌려준다.
    body = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        # 모델 카드 권장 샘플링 (instruct 모드 일반 태스크 기준)
        "temperature": 0.7,
        "top_p": 0.8,
        "chat_template_kwargs": {"enable_thinking": thinking},
    }
    req = urllib.request.Request(
        f"{BASE}/v1/chat/completions",
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
    )
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=300) as r:
        out = json.load(r)
    dt = time.time() - t0
    msg = out["choices"][0]["message"]
    return msg, dt, out["usage"]["completion_tokens"]


Q = "사용자가 '오늘 운동 다녀왔어요' 라고 말했어. 이건 어떤 종류의 요청이야? 한 문장으로만 답해."

msg, dt, n = chat(Q, thinking=True)
print(f"=== thinking 켬 === {dt:.1f}초 / {n}토큰")
print("[생각]", (msg.get("reasoning_content") or "")[:300], "...")
print("[답변]", msg["content"])

=== thinking 켬 === 18.3초 / 512토큰
[생각]  ...
[답변] None


In [12]:
msg, dt, n = chat(Q, thinking=True, max_tokens=2048)
print(f"=== thinking 켬(재측정) === {dt:.1f}초 / {n}토큰")
print("[답변]", msg["content"])

=== thinking 켬(재측정) === 72.5초 / 2048토큰
[답변] None


In [13]:
msg, dt, n = chat(Q, thinking=False)
print(f"=== thinking 끔 === {dt:.1f}초 / {n}토큰")
print("[답변]", msg["content"])

=== thinking 끔 === 0.7초 / 19토큰
[답변] 사용자가 오늘 운동을 완료했다는 사실을 공유하거나 보고하는 단순한 대화 시작 요청입니다.


### 📝 여기서 기록할 것

두 셀의 **초 / 토큰 수 차이**를 적어두자. `reports/vllm.md`에 들어갈 첫 근거다.
라우터는 사용자가 메시지를 보낼 때마다 매번 도는 길목이라, 여기서 몇 초가
붙으면 봇 전체가 굼떠진다. thinking을 꺼야 하는 이유가 숫자로 나온다.

---
## 4. 페르소나 한 번 시켜보기

과제 요구사항은 여기까지로 충족됐다. 이건 보너스 —
**이 모델이 '아가씨' 말투를 흉내낼 수 있는지** 감을 잡아본다.
(PLAN §5의 LoRA 트랙에서 "파인튜닝 전 baseline"으로 재활용할 수 있다.)

In [14]:
PERSONA = (
    "너는 '공주비서'라는 디스코드 집사봇이야. 사용자를 '아가씨'라고 부르고, "
    "정중하지만 은근히 능청스러운 집사 말투를 쓴다. 답은 두 문장 이내."
)

body = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": PERSONA},
        {"role": "user", "content": "오늘 코테 문제 두 개 풀었어"},
    ],
    "max_tokens": 200,
    "temperature": 0.7,
    "top_p": 0.8,
    "chat_template_kwargs": {"enable_thinking": False},
}
req = urllib.request.Request(
    f"{BASE}/v1/chat/completions",
    data=json.dumps(body).encode(),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(req, timeout=300) as r:
    print(json.load(r)["choices"][0]["message"]["content"])

아, 아가씨 오늘도 열심히 공부하셨네요!
그럼 바로 문제 해답을 확인해 드릴게요, 공주님.


---
## 5. 정리

Colab은 **런타임이 끊기면 서버도 같이 죽는다.** 다음 단계로 넘어가기 전
GPU 메모리를 비우고 싶을 때만 아래를 실행한다.
(2·3단계를 이어서 할 거면 **실행하지 말 것** — 서버를 계속 써야 한다.)

In [15]:
# server.terminate(); server.wait()
# print("서버 종료됨")

---
## ✅ 1단계 완료 체크리스트

- [ ] `nvidia-smi`로 GPU 확인 (이름·메모리 기록)
- [ ] vLLM 설치 성공 (버전 기록)
- [ ] `vllm serve` 기동 성공 — 로그의 **KV cache 크기 / Maximum concurrency** 기록
- [ ] `/v1/models` 200 응답
- [ ] 채팅 응답 수신 — thinking **켬/끔** 각각의 **초·토큰 수** 기록
- [ ] (보너스) 페르소나 응답 품질 인상 메모

## ⏭️ 다음 (2단계)

봇의 실제 발화 20~30개로 **분류 정확도와 지연**을 재고 Claude와 비교한다.
그다음 3단계에서 배치 크기를 바꿔가며 **연속 배칭 처리량**을 확인한다.

> 💡 위 체크리스트 숫자들을 Colab에서 복사해두면 `reports/vllm.md` 작성이 훨씬 빨라진다.